# [Super AI Engineer Season 6] Hackathon Week 6
## 5 Domains Hackathon: Chest Disease Prediction

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Chest Disease Detection
- Domain: Computer Vision / Medical Image Classification
- Notebook: Advanced pipeline using EfficientNet-B4 + Focal Loss + SOTA Augmentations
- จัดทำโดย: 600425-วิศิษฐ์

---
### Improvement for this notebook
1. **Backbone Upgrade**: เปลี่ยนจาก `DenseNet121` เป็น `EfficientNet-B4` ซึ่งมีความสามารถในการ extract feature ที่ดีกว่า
2. **Image Resolution**: เพิ่มขนาดภาพจาก `224x224` เป็น `384x384` เพื่อให้สามารถตรวจจับรายละเอียดเล็กๆ ในภาพ X-ray ได้
3. **Loss Function**: เปลี่ยนมาใช้ `Focal Loss` สำหรับรับมือกับปัญหา Class Imbalance ที่พบมากใน Medical Imaging
4. **Augmentations**: เพิ่ม `ColorJitter` และ `RandomAffine` เพื่อเสริมความแข็งแกร่ง (Robustness) ของโมเดล
5. **Scheduler**: ใช้ `CosineAnnealingWarmRestarts` ช่วยให้ Optimizer ปรับตัวได้ดีขึ้น

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. Dataset & Advanced Augmentation  
4. Train/Validation Split  
5. Model Preparation (EfficientNet-B4)  
6. Training & Validation (Focal Loss) 
7. Inference with TTA  
8. Prediction & Submission Generation  
9. Summary

# 1. Setup & Imports
### 1.1 Prepare the environment and core settings

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ─── SEED ───────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─── CONFIG ─────────────────────────────────────────────────────────────────
KAGGLE_MODE = Path('/kaggle/input').exists()
WORK_DIR    = Path('/kaggle/working') if KAGGLE_MODE else Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR_CANDIDATES = [
    Path('/kaggle/input/competitions/chest-disease-detection'),
    Path('/kaggle/input/individual-test-chest-disease-detection'),
    Path('/content'),
    Path('.'),
]
data_dir = next((p for p in DATA_DIR_CANDIDATES if p.exists()), Path('.'))

# Training hyperparameters (v3 Updates)
IMG_SIZE    = 384          # v3 Update: เพิ่มขนาดรูปเป็น 384 (EfficientNet-b4 native ~380)
BATCH_SIZE  = 16           # v3 Update: ลด Batch Size ลงมาเพื่อกัน OOM จากรูปใหญ่
NUM_EPOCHS  = 7            # v3 Update: รัน 7 Epochs กับ WarmRestarts
LR_HEAD     = 1e-3         
LR_BACKBONE = 1e-4         
NUM_WORKERS = 2
USE_TTA     = True         
TTA_N       = 5            

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_COLUMNS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
    'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax'
]
NUM_CLASSES = len(LABEL_COLUMNS)

submission_path = WORK_DIR / '5hack_chest_disease_v3_submission.csv'
model_path      = WORK_DIR / 'effnetb4_chest_best_v3.pth'

print(f'Device : {DEVICE}')
print(f'Kaggle : {KAGGLE_MODE}')
print(f'Data   : {data_dir.resolve()}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# 2. Data Loading & Initial Inspection


In [ ]:
def find_file(root: Path, names):
    for name in names:
        p = root / name
        if p.exists():
            return p
    for name in names:
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Cannot find {names} under {root}')

def find_image_dir(root: Path):
    for subdir in ['train_images', 'images', 'train', '']:
        d = root / subdir if subdir else root
        if d.is_dir():
            jpegs = list(d.glob('*.jpg'))
            if jpegs:
                return d
    for d in root.rglob('*'):
        if d.is_dir() and list(d.glob('*.jpg')):
            return d
    return root

train_path      = find_file(data_dir, ['train.csv'])
sample_sub_path = find_file(data_dir, ['sample_submission.csv', 'test_submission.csv'])

train_df  = pd.read_csv(train_path)
sample_df = pd.read_csv(sample_sub_path)

for df in [train_df, sample_df]:
    for col in ['filename', 'image_name', 'id', 'image_id', 'Image']:
        if col in df.columns:
            df.rename(columns={col: 'image_name'}, inplace=True)
            break

train_img_dir = find_image_dir(data_dir / 'train_images' if (data_dir / 'train_images').exists() else data_dir)
test_img_dir  = find_image_dir(data_dir / 'test_images'  if (data_dir / 'test_images').exists()  else data_dir)

train_labeled = train_df.dropna(subset=LABEL_COLUMNS).copy()
test_df = sample_df[['image_name']].copy()

print(f'Labeled train rows : {len(train_labeled)}')
print(f'Test rows          : {len(test_df)}')

# 3. Dataset & Advanced Augmentation
เพิ่ม ColorJitter และ RandomAffine เพื่อช่วยโมเดลรับมือกับภาพ X-ray ทรงแปลกๆ และ contrast ไม่คงที่

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), # v3 Update: เพิ่ม degree จาก 10 เป็น 15
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)), # v3 Update: Affine transform
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tta_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ChestDataset(Dataset):
    def __init__(self, df, img_dir, label_cols, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.label_cols= label_cols
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.img_dir / row['image_name']
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, row['image_name']
        labels = torch.FloatTensor(row[self.label_cols].values.astype(float))
        return img, labels

# 4. Train/Validation Split


In [ ]:
strat_key = train_labeled[LABEL_COLUMNS].sum(axis=1).clip(0, 4).astype(int)

train_idx, val_idx = train_test_split(
    range(len(train_labeled)),
    test_size=0.1,
    random_state=SEED,
    stratify=strat_key,
)

df_train = train_labeled.iloc[train_idx].reset_index(drop=True)
df_val   = train_labeled.iloc[val_idx].reset_index(drop=True)

train_ds = ChestDataset(df_train, train_img_dir, LABEL_COLUMNS, train_transform)
val_ds   = ChestDataset(df_val,   train_img_dir, LABEL_COLUMNS, val_transform)
test_ds  = ChestDataset(test_df,  test_img_dir,  LABEL_COLUMNS, val_transform, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# 5. Model Preparation (EfficientNet-B4) & Focal Loss
จากเดิมแทนที่จะใช้ `DenseNet121` เราเปลี่ยนเป็น `EfficientNet-B4` ซึ่งให้ความแม่นยำสูงกว่าบน ImageNet และ SOTA ในหลายงาน

In [ ]:
def build_model(num_classes=13):
    model = models.efficientnet_b4(weights='IMAGENET1K_V1')
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

model = build_model(NUM_CLASSES).to(DEVICE)

# v3 Update: Focal Loss Implementation (แก้ปัญหา Long-tailed data ได้ดีกว่า pos_weight ธรรมดา)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

criterion = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = optim.Adam([
    {'params': model.features.parameters(),    'lr': LR_BACKBONE},
    {'params': model.classifier.parameters(),  'lr': LR_HEAD},
], weight_decay=1e-4)

# v3 Update: CosineAnnealingWarmRestarts
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2, eta_min=1e-6)

# 6. Training & Validation


In [ ]:
def compute_mean_auc(labels_all, preds_all, label_cols):
    aucs = []
    for i, lbl in enumerate(label_cols):
        y_true = labels_all[:, i]
        y_pred = preds_all[:, i]
        if y_true.sum() == 0 or (1 - y_true).sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true, y_pred))
    return float(np.mean(aucs)), aucs

def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    losses = []
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        # scheduler.step() ถ้าเป็น OneCycleLR
    return float(np.mean(losses))

def validate(model, loader, criterion, device, label_cols):
    model.eval()
    losses    = []
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Val  ', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            losses.append(loss.item())
            preds  = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    all_preds  = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    mean_auc, per_class = compute_mean_auc(all_labels, all_preds, label_cols)
    return float(np.mean(losses)), mean_auc, per_class

In [ ]:
best_auc   = 0.0
history    = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\n{'='*60}')
    print(f'Epoch {epoch}/{NUM_EPOCHS}')

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, DEVICE)
    val_loss, val_auc, per_class_auc = validate(model, val_loader, criterion, DEVICE, LABEL_COLUMNS)
    scheduler.step() # สำหรับ WarmRestarts step หลังจบ epoch

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_auc': val_auc})

    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Val Loss   : {val_loss:.4f}')
    print(f'  Val AUC    : {val_auc:.4f}  {"★ NEW BEST" if val_auc > best_auc else ""}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), model_path)
        print(f'  Saved best model → {model_path}')

print(f'\nBest Validation AUC: {best_auc:.4f}')

# 7. Inference with TTA


In [ ]:
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

all_filenames = []
all_preds     = []

if USE_TTA:
    tta_datasets = [
        ChestDataset(test_df, test_img_dir, LABEL_COLUMNS, tta_transform, is_test=True)
        for _ in range(TTA_N)
    ]
    tta_loaders = [
        DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        for ds in tta_datasets
    ]

    tta_preds_list = []
    for tta_i, loader in enumerate(tta_loaders):
        preds_i   = []
        fnames_i  = []
        with torch.no_grad():
            for imgs, fnames in tqdm(loader, desc=f'TTA {tta_i+1}/{TTA_N}', leave=False):
                imgs = imgs.to(DEVICE)
                logits = model(imgs)
                probs  = torch.sigmoid(logits).cpu().numpy()
                preds_i.append(probs)
                fnames_i.extend(fnames)
        tta_preds_list.append(np.concatenate(preds_i, axis=0))
        if tta_i == 0:
            all_filenames = fnames_i

    all_preds = np.mean(tta_preds_list, axis=0)

else:
    with torch.no_grad():
        for imgs, fnames in tqdm(test_loader, desc='Inference'):
            imgs  = imgs.to(DEVICE)
            logits= model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_filenames.extend(fnames)
    all_preds = np.concatenate(all_preds, axis=0)

# 8. Prediction & Submission Generation


In [ ]:
sub_df = pd.DataFrame({'filename': all_filenames})
for i, col in enumerate(LABEL_COLUMNS):
    sub_df[col] = all_preds[:, i]

assert sub_df.shape[0] == sample_df.shape[0]
sub_df.to_csv(submission_path, index=False)
print(f'\nSaved → {submission_path}')